In [23]:
!pip install langchain-ollama

In [1]:
import requests

In [5]:
from langchain_core.tools import tool

In [25]:
@tool
def get_weather(ziel: str, datum_von: str , datum_bis: str)->dict:
    """Wettervorhersage"""
    r = requests.get(
        "https://api.open-meteo.com/v1/forecast?latitude=52.516181&longitude=13.3773359&daily=temperature_2m_max,temperature_2m_min,precipitation_probability_max&timezone=auto",
        params={
            "latitude":52.516181,
            "longitude":13.3773359,
            "daily":"temperature_2m_max,temperature_2m_min,precipitation_probability_max",
            "timezone":"auto",
        },
        timeout=10,
    )
    r.raise_for_status()
    return r.json()

In [31]:
#Agent
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

llm = init_chat_model("ollama:llama3.2 ")

weather_agent = create_agent(
    llm,
    tools=[get_weather],
    system_prompt=(
        "Du bist ein Wetter-Agent. Rufe get_weather auf und antworte kurz: "
        "Tempatur, Regen , die Sonne scheint und/oder nicht je Tag"
    ),
)

In [33]:
#Test
res = weather_agent.invoke(
    {"message":[{"role":"user","content":"Wetter in Berlim am 15.11.2026"}]}
)
print(res["messages"])

[AIMessage(content='assistant\n\nIch bin bereit, dir zu helfen! Bitte gib mir den Name des Tools, das du aufrufen möchtest.', additional_kwargs={}, response_metadata={'model': 'llama3.2 ', 'created_at': '2026-11-14T08:57:34.9864927Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1030673300, 'load_duration': 31500700, 'prompt_eval_count': 83, 'prompt_eval_duration': 311353000, 'eval_count': 31, 'eval_duration': 679619000, 'logprobs': None, 'model_name': 'llama3.2 ', 'model_provider': 'ollama'}, id='lc_run--01a1d946-bbff-751d-8190-3f00a648dfc5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 31, 'total_tokens': 114})]
